# Logistic Regression and Classification

In our previous notebook, we explored **Linear Regression** to predict continuous values like taxi fares. The output of a linear regression model can be any real number (from negative infinity to positive infinity).

But what if we want to predict a category instead of a continuous number? This is called **Classification**.
* **Binary Classification**: Predicting one of two possible categories (e.g., "Will it rain tomorrow?" -> Yes or No; "Is this email spam?" -> Spam or Not Spam).
* We typically represent the categories as `1` (Yes/Spam) and `0` (No/Not Spam).

### Why Linear Regression Fails for Classification
If we use our linear model equation $y = wx + b$ for classification, the model will draw a straight line.
1. The line can predict values like `1.5` or `-0.5`. What does a prediction of `1.5` mean when the only valid options are `0` or `1`? It makes no sense!
2. We need a model that outputs a **probability** (a value strictly between `0` and `1`) so we can interpret it as "there is an 80% chance of rain tomorrow".

### Entering Logistic Regression
This is where **Logistic Regression** comes in. Despite its name, Logistic Regression is actually a **classification** algorithm, not a regression algorithm.

It helps us by predicting the **probability** that a given input belongs to a specific category (e.g., a 75% probability that it will rain).

To do this, it takes our linear regression equation $z = wx + b$ (which can output any value) and maps it into a probability between 0 and 1 using the **Sigmoid Function**.

### Entering the Sigmoid Function
To solve this, we take the output of our linear equation (let's call it $z$):
$$z = w_1x_1 + w_2x_2 + ... + b$$

And we pass it through a mathematical function called the **Sigmoid Function** (also known as the Logistic Function), represented by $\sigma(z)$:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

This function has an S-shaped curve that perfectly "squishes" any input value $z$ into a value between `0` and `1`:
* **Extreme Positive Input ($z \to \infty$)**: $e^{-z}$ becomes almost $0$, so $\sigma(z) \approx \frac{1}{1 + 0} = 1$.
* **Extreme Negative Input ($z \to -\infty$)**: $e^{-z}$ becomes extremely large, making the denominator huge. So $\sigma(z) \approx \frac{1}{\infty} = 0$.
* **Zero Input ($z = 0$)**: $e^0 = 1$, so $\sigma(0) = \frac{1}{1 + 1} = 0.5$ (exactly a 50% chance).


In [ ]:
# @title Coding Example: How Sigmoid Helps Us Categorize
import numpy as np

# Let's define the Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Problem: Will a student pass the exam based on hours studied?
# Features (x): Hours studied
# Let's assume we already trained a model and found these parameters:
weight = 0.8  # weight (w)
bias = -3.0   # bias (b)

# Let's test for different study hours
study_hours = np.array([1, 3, 4, 6, 8])

# Step 1: Calculate the linear output (z)
z_values = (weight * study_hours) + bias

# Step 2: Pass through Sigmoid to get probabilities (y_predicted)
probabilities = sigmoid(z_values)

# Step 3: Categorize (Thresholding at 0.5: >= 0.5 is Pass (1), else Fail (0))
predictions = (probabilities >= 0.5).astype(int)

# Print the results clearly
print(f"{'Hours Studied':<15} | {'Linear Output (z)':<18} | {'Probability (y\')':<18} | {'Final Prediction':<15}")
print("-" * 75)
for h, z, prob, pred in zip(study_hours, z_values, probabilities, predictions):
    status = "Pass (1)" if pred == 1 else "Fail (0)"
    print(f"{h:<15} | {z:<18.2f} | {prob:<18.4f} | {status:<15}")


Hours Studied   | Linear Output (z)  | Probability (y')   | Final Prediction
---------------------------------------------------------------------------
1               | -2.20              | 0.0998             | Fail (0)       
3               | -0.60              | 0.3543             | Fail (0)       
4               | 0.20               | 0.5498             | Pass (1)       
6               | 1.80               | 0.8581             | Pass (1)       
8               | 3.40               | 0.9677             | Pass (1)       


## 📉 2. Key Concepts: Log Loss vs. Mean Squared Error (MSE)

When training our model, we need a way to calculate **how wrong** the model's guesses are. We call this **Loss**.

In **Linear Regression**, we used **Mean Squared Error (MSE)**, which calculates the average squared difference between the actual taxi fare and the predicted fare.

However, for **Logistic Regression**, we use **Log Loss** instead. Here is why MSE doesn't work well here in a simple way:

 MSE makes learning too slow when the model is wildly wrong:
*   Because Logistic Regression uses the Sigmoid function to force outputs between `0` and `1`, the curve becomes extremely flat at the very ends (near `0` and `1`).
*   If the model is highly confident but completely wrong (e.g., it predicts a `99%` chance of rain, but it turns out to be a sunny day), MSE's mathematical curve becomes so flat that the model barely learns from its mistake. It gets "stuck" and updates its weights at a snail's pace.
*   **Log Loss** fixes this by using a logarithmic scale. If the model is highly confident but wrong, Log Loss shoots up to infinity, giving the model a massive "nudge" to quickly correct itself.



## 🛡️ 3. Regularization and the "Dog on a Leash" Analogy

Overfitting happens when a model memorizes details of the training data instead of learning general patterns. To prevent this, we use a technique called **Regularization** (specifically **L2 Regularization** or Ridge).

To understand this simply, let's use an analogy:

### 🐕 The Dog on a Leash Analogy

Imagine you are training a puppy (our model) to stay on a sidewalk (the decision boundary between Pass/Fail or 0/1):

1.  **Runaway Weights (The Wild Dog without a Leash)**:
    *   If the training data is **perfectly separable** (e.g., all students who studied $<3$ hours failed, and all who studied $\ge 4$ hours passed), the puppy discovers that the further it runs away from the sidewalk, the more "correct" its predictions are.
    *   The Sigmoid function wants to predict exactly `100%` or `0%`. But Sigmoid can only do this if the input is **infinity**.
    *   Without a leash, the model will continuously inflate the weight ($w$) during training, trying to reach infinity. The weight "runs away"!
    *   **Why this is bad**: The model becomes extremely overconfident. If a new student studies 3.9 hours, the model might panic and make a wildly wrong prediction because its weights are unnaturally huge.

2.  **Regularization (The Leash)**:
    *   Regularization acts like a **leash** on the dog.
    *   It tells the model: "You want to be correct, but if you run too far (if your weights grow too large), you will receive a penalty score."
    *   This penalty forces the model to keep its weights small and sensible, making it much more robust and prepared for new, slightly different data.


In [6]:
# @title Coding Example: Regularization in Action (Leash vs. No Leash)
from sklearn.linear_model import LogisticRegression
import numpy as np

# Perfectly linearly separable dataset
# Hours studied: 1, 2 (fail, y=0) vs 5, 6 (pass, y=1)
X = np.array([1, 2, 5, 6]).reshape(-1, 1)
y = np.array([0, 0, 1, 1])

# Model A: NO LEASH (Almost no regularization, C is huge)
model_no_leash = LogisticRegression(C=1e6, solver='lbfgs')
model_no_leash.fit(X, y)

# Model B: STRONG LEASH (Strong regularization, C is small)
model_strong_leash = LogisticRegression(C=0.1, solver='lbfgs')
model_strong_leash.fit(X, y)

# Print comparison
print(f"{'Model Type':<25} | {'Learned Weight (w)':<20} | {'Base Fee / Bias (b)':<20}")
print("-" * 75)
print(f"{'Model A (No Leash)':<25} | {model_no_leash.coef_[0][0]:<20.4f} | {model_no_leash.intercept_[0]:<20.4f}")
print(f"{'Model B (Strong Leash)':<25} | {model_strong_leash.coef_[0][0]:<20.4f} | {model_strong_leash.intercept_[0]:<20.4f}")

print("\nObservations:")
print("- Model A (No Leash) has an extremely large weight, making its decision boundary highly rigid and overconfident.")
print("- Model B (Strong Leash) keeps the weight small and sensible, making it much safer for predicting on new data!")


Model Type                | Learned Weight (w)   | Base Fee / Bias (b) 
---------------------------------------------------------------------------
Model A (No Leash)        | 6.3032               | -21.1475            
Model B (Strong Leash)    | 0.2836               | -0.9927             

Observations:
- Model A (No Leash) has an extremely large weight, making its decision boundary highly rigid and overconfident.
- Model B (Strong Leash) keeps the weight small and sensible, making it much safer for predicting on new data!


## 🔄 4. Training Loss, Validation Loss, and Early Stopping

When we train a machine learning model, we want it to perform well on **new, unseen data**, not just memorize the examples we show it. To check this, we split our data into two main sets:

### 📚 The Exam Analogy

Imagine a student preparing for a history exam:
1.  **Training Loss (Practice Questions)**:
    *   These are the practice questions the student studies over and over again.
    *   **Training Loss** measures the errors on these study questions. As the student spends more time studying, their score on these questions will naturally get better and better (Training Loss goes down).
2.  **Validation Loss (Mock Test)**:
    *   This is a fresh mock test with brand-new questions the student has never seen before.
    *   **Validation Loss** measures the errors on this mock test. This tells us if the student actually *understands* history, or if they have just *memorized* the answers to the practice questions.

---

### 📈 Overfitting and Early Stopping

*   **Early Phase**: At the start of studying, both Training Loss and Validation Loss go down. The student is learning general rules.
*   **Overfitting Phase**: Eventually, if the student studies *too* much, they will start memorizing the exact punctuation and typos of the practice questions. Their score on the practice questions will be perfect (Training Loss continues to drop to near $0$), but their score on the mock test will get **worse** (Validation Loss starts to rise). This is called **Overfitting**.

### How Early Stopping Helps
**Early Stopping** is a smart safety switch. During training, we monitor the Validation Loss after each study round (epoch). The moment the Validation Loss stops decreasing and begins to rise, we **stop training immediately**.

This ensures we keep the model at its peak performance on new data, before it has a chance to overfit and memorize the training set.


## 📊 5. Making Decisions: Classification Threshold & Confusion Matrix

Logistic regression predicts the **probability** that a given day will be rainy (a value between `0.0` and `1.0`). However, in the real world, we need a yes/no decision: **"Should I carry an umbrella?"**

To convert a probability into a Yes (1) or No (0) class prediction, we use a **Classification Threshold**.

### 1. The Threshold (Default = 0.5)
By default, if the predicted probability is $\ge 0.5$, we classify it as **Rain (1)**. Otherwise, it is **No Rain (0)**. 

However, we can adjust this threshold depending on our risk tolerance:
*   **Low Threshold (e.g., 0.2)**: We predict Rain even if there is a small 20% chance. We will catch almost all rainy days, but we'll carry an umbrella on many dry days (False Alarms).
*   **High Threshold (e.g., 0.7)**: We only predict Rain if we are highly confident. We'll rarely carry an umbrella on dry days, but we might get soaked on days with a 60% chance of rain (Missed Days).

---

### 2. The Confusion Matrix
To evaluate our predictions under a selected threshold, we use a **Confusion Matrix**. It is a simple $2 \times 2$ grid that compares the model's guesses against the actual outcomes:

| | Actual: No Rain (0) | Actual: Rain (1) |
| :--- | :--- | :--- |
| **Predicted: No Rain (0)** | **True Negative (TN)** <br> (Correctly predicted No Rain) | **False Negative (FN)** <br> (Predicted dry, but it rained! 🌧️) |
| **Predicted: Rain (1)** | **False Positive (FP)** <br> (Predicted rain, but it was dry! ☀️) | **True Positive (TP)** <br> (Correctly predicted Rain) |

Let's see how shifting the threshold changes this matrix in the coding example below!


In [ ]:
# @title Coding Example: Confusion Matrix at Different Thresholds
from sklearn.metrics import confusion_matrix
import numpy as np

# Simulate actual outcomes (y_true): 3 dry days, 5 rainy days
y_true = np.array([0, 0, 0, 1, 1, 1, 1, 1])

# Simulated predicted probabilities of rain
y_probs = np.array([0.1, 0.35, 0.6, 0.4, 0.7, 0.8, 0.9, 0.95])

# Function to print confusion matrix nicely
def print_cm(cm, threshold):
    print(f"--- Confusion Matrix (Threshold = {threshold}) ---")
    print(f"{'':<18} | {'Actual: 0 (Dry)':<15} | {'Actual: 1 (Rain)':<15}")
    print("-" * 55)
    print(f"{'Predicted: 0 (Dry)':<18} | {cm[0][0]:<15} | {cm[0][1]:<15} (TN / FN)")
    print(f"{'Predicted: 1 (Rain)':<18} | {cm[1][0]:<15} | {cm[1][1]:<15} (FP / TP)\n")

# 1. Default Threshold = 0.5
preds_50 = (y_probs >= 0.5).astype(int)
cm_50 = confusion_matrix(y_true, preds_50)
print_cm(cm_50, 0.5)

# 2. High Threshold = 0.7
preds_70 = (y_probs >= 0.7).astype(int)
cm_70 = confusion_matrix(y_true, preds_70)
print_cm(cm_70, 0.7)


## 📈 6. Evaluating the Model: Classification Metrics

A single confusion matrix shows us our raw counts, but we need standard numbers to measure our model's overall quality. Here are the four key classification metrics:

### 1. Accuracy (Are we generally right?)
The percentage of total correct guesses.
$$\text{Accuracy} = \frac{\text{True Positives (TP)} + \text{True Negatives (TN)}}{\text{Total Examples}}$$
*Caveat*: If $99\%$ of days are sunny, a model that always guesses "Sunny" is $99\%$ accurate but completely useless for predicting rain. This is why we need other metrics.

### 2. Precision (How trustworthy are our "Rain" predictions?)
Of all the times we predicted Rain, how often was it actually raining?
$$\text{Precision} = \frac{\text{True Positives (TP)}}{\text{True Positives (TP)} + \text{False Positives (FP)}}$$
*Use Case*: High precision is vital if carrying an umbrella is a massive hassle. We only predict rain when we are absolutely certain (minimizes False Positives).

### 3. Recall / Sensitivity (Did we catch all the rainy days?)
Of all the days it actually rained, how many did we successfully catch?
$$\text{Recall} = \frac{\text{True Positives (TP)}}{\text{True Positives (TP)} + \text{False Negatives (FN)}}$$
*Use Case*: High recall is vital if getting wet makes you sick. You carry an umbrella even on low-chance days (minimizes False Negatives).

### 4. F1-Score (The Balance)
The harmonic average of Precision and Recall. It gives you a single score that balances both.
$$\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$


In [ ]:
# @title Coding Example: Calculating Metrics for Threshold 0.5 vs. 0.7
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Calculate metrics for Threshold = 0.5
acc_50 = accuracy_score(y_true, preds_50)
prec_50 = precision_score(y_true, preds_50)
rec_50 = recall_score(y_true, preds_50)
f1_50 = f1_score(y_true, preds_50)

# Calculate metrics for Threshold = 0.7
acc_70 = accuracy_score(y_true, preds_70)
prec_70 = precision_score(y_true, preds_70)
rec_70 = recall_score(y_true, preds_70)
f1_70 = f1_score(y_true, preds_70)

# Print comparison table
print(f"{'Metric':<15} | {'Threshold = 0.5':<18} | {'Threshold = 0.7':<18}")
print("-" * 55)
print(f"{'Accuracy':<15} | {acc_50:<18.4f} | {acc_70:<18.4f}")
print(f"{'Precision':<15} | {prec_50:<18.4f} | {prec_70:<18.4f} (Increases with higher threshold!)")
print(f"{'Recall':<15} | {rec_50:<18.4f} | {rec_70:<18.4f} (Decreases with higher threshold!)")
print(f"{'F1-Score':<15} | {f1_50:<18.4f} | {f1_70:<18.4f}")


## 📉 7. The Overall Score: ROC Curve and AUC

How do we know if our model is good overall, regardless of which threshold we choose? We use the **ROC Curve** and **AUC** score.

### 1. The ROC Curve (Receiver Operating Characteristic)
The ROC curve is a graph that shows how our model behaves at **every possible threshold** from `0.0` to `1.0`.
*   **Vertical Axis**: True Positive Rate (Recall) -> The proportion of rainy days we successfully catch.
*   **Horizontal Axis**: False Positive Rate -> The proportion of dry days we incorrectly flag as rainy (False Alarms).

A good model's curve arches high towards the top-left corner (meaning high Recall with very few False Alarms).

### 2. AUC (Area Under the Curve)
AUC is the entire area underneath the ROC curve. It is a single score between `0.0` and `1.0` that summarizes the model's quality:
*   **AUC = 1.0**: A perfect model. It makes zero mistakes.
*   **AUC = 0.5**: A useless model. It is no better than guessing randomly (flipping a coin).


In [ ]:
# @title Coding Example: Plotting the ROC Curve & Calculating AUC
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Calculate ROC curve points
fpr, tpr, thresholds = roc_curve(y_true, y_probs)

# Calculate AUC
auc_score = roc_auc_score(y_true, y_probs)

# Plot ROC Curve
plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, color="teal", lw=2, label=f"ROC Curve (AUC = {auc_score:.4f})")
plt.plot([0, 1], [0, 1], color="grey", linestyle="--", label="Random Guess (AUC = 0.5)")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (False Alarm Rate)")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve: Measuring Overall Classification Quality")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Overall Model Quality (AUC Score): {auc_score:.4f}")


## ⚖️ 8. Prediction Bias

In machine learning, **Prediction Bias** measures whether our model is systematically over-predicting or under-predicting compared to reality. 

It is defined as:
$$\text{Prediction Bias} = \text{Average of Predicted Probabilities} - \text{Average of Actual Labels (Real Frequency)}$$

*   **Zero Bias (Ideal)**: If our average predicted probability of rain is $62\%$, and it actually rained on $62\%$ of the days in our dataset, our prediction bias is $0$.
*   **Positive Bias (Over-predicting)**: The model predicts a high probability of rain more often than it actually rains.
*   **Negative Bias (Under-predicting)**: The model predicts a low probability of rain more often than it actually rains.


In [ ]:
# @title Coding Example: Calculating Prediction Bias
import numpy as np

# Calculate average predicted probability
avg_prediction = np.mean(y_probs)

# Calculate average actual label (real frequency in our dataset)
real_frequency = np.mean(y_true)

# Bias
bias = avg_prediction - real_frequency

print(f"Average Predicted Probability: {avg_prediction:.4f}")
print(f"Actual Real-World Frequency:   {real_frequency:.4f}")
print(f"Prediction Bias:               {bias:+.4f}")


## 🌈 9. Multi-Class Classification

So far, we have explored **Binary Classification** (Rain / No Rain). But what if we want to predict one of **three or more** categories? For example, predicting whether today is **Sunny (0)**, **Rainy (1)**, or **Snowy (2)**?

This is called **Multi-Class Classification**.

### How it Works (One-vs-All / OvA)
Instead of reinventing the wheel, the model trains multiple binary classifiers behind the scenes:
1.  **Classifier 1**: Sunny vs. Not Sunny
2.  **Classifier 2**: Rainy vs. Not Rainy
3.  **Classifier 3**: Snowy vs. Not Snowy

When you ask the model to classify a day, it runs all three classifiers and picks the one with the highest probability.

### The Softmax Function
Instead of the Sigmoid function (which only squishes one linear output between 0 and 1), Multi-Class models use the **Softmax Function**.
*   Softmax calculates the probabilities of all classes simultaneously.
*   It ensures that all predicted probabilities across all classes sum up to exactly **1.0** (e.g., 70% Sunny, 20% Rainy, 10% Snowy).


In [ ]:
# @title Coding Example: Multi-Class Classification with Scikit-Learn
from sklearn.linear_model import LogisticRegression
import numpy as np

# Toy dataset for 3 classes (Sunny=0, Rainy=1, Snowy=2)
# Features: [Temperature, Humidity]
X_multi = np.array([
    [30, 20], [28, 15], # Hot, Dry -> Sunny (0)
    [15, 85], [18, 90], # Mild, Humid -> Rainy (1)
    [-5, 40], [-2, 50]  # Freezing -> Snowy (2)
])
y_multi = np.array([0, 0, 1, 1, 2, 2])

# Train a multi-class logistic regression model
multi_model = LogisticRegression(solver='lbfgs')
multi_model.fit(X_multi, y_multi)

# Test on a new day: 16°C and 88% Humidity (looks like rain!)
new_day = np.array([[16, 88]])
prediction = multi_model.predict(new_day)
probabilities = multi_model.predict_proba(new_day)[0]

# Print class probabilities
classes = ["Sunny (0)", "Rainy (1)", "Snowy (2)"]
print("Predicted Probabilities:")
for c_name, prob in zip(classes, probabilities):
    print(f"- {c_name}: {prob:.4f}")

print(f"\nFinal Predicted Category: {classes[prediction[0]]}")


### Libraries Used in this Project

*   **Pandas:** Think of this as "Excel for Python." It helps us organize and clean data using tables called DataFrames.
*   **NumPy:** Used for fast mathematical calculations and handling large lists of numbers (arrays).
*   **Scikit-Learn (sklearn):** The primary toolbox for Machine Learning. It contains the algorithms and tools needed to train and test models.
    *   `train_test_split`: Helps us hide some data from the model so we can test it later.
    *   `LogisticRegression`: The "brain" that learns the patterns in the weather data.
    *   `accuracy_score`: A simple way to see what percentage of our predictions were correct.

### What are we doing in this exercise?

In this exercise, we are performing **Binary Classification**.

**The Goal:** We want to look at today's weather patterns (like Temperature, Humidity, and Pressure) and predict a "Yes" or "No" answer to the question: **"Will it rain tomorrow?"**

We use **Logistic Regression** because it is a great algorithm for predicting categories (like Yes/No). It looks at historical data, finds correlations between weather factors and rainfall, and creates a mathematical boundary to classify new weather data.

## Australian Rainfall Prediction

**Objective:** Build a classification model using Logistic Regression to predict whether it will rain tomorrow in Australia based on weather observations.

## Step 1: Download the Dataset

The dataset is sourced from Kaggle ('Rain in Australia'). Since we are running in **Google Colab**, we download it directly from our GitHub repository using a URL — no manual upload needed!

> **Note:** The full dataset (~145,000 rows) is available on [Kaggle](https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package). If you have a Kaggle account, you can upload your `kaggle.json` API key to Colab and use the Kaggle API to download it directly.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# -------------------------------------------------------------------
# Load data directly from GitHub (works in Colab without any upload!)
# The dataset is the 'Rain in Australia' dataset from Kaggle.
# -------------------------------------------------------------------
DATA_URL = 'https://raw.githubusercontent.com/akshaykhanna/learning-complete-ai/main/weather.csv'

df = pd.read_csv(DATA_URL)

# Print shape and first 5 rows
print("Data Shape:", df.shape)
print(df.head())

Data Shape: (145460, 24)
         Date Location  MinTemp  MaxTemp  Rainfall  Evaporation  Sunshine  \
0  2008-12-01   Albury     13.4     22.9       0.6          NaN       NaN   
1  2008-12-02   Albury      7.4     25.1       0.0          NaN       NaN   
2  2008-12-03   Albury     12.9     25.7       0.0          NaN       NaN   
3  2008-12-04   Albury      9.2     28.0       0.0          NaN       NaN   
4  2008-12-05   Albury     17.5     32.3       1.0          NaN       NaN   

  WindGustDir  WindGustSpeed WindDir9am  ... Humidity3pm  Pressure9am  \
0           W           44.0          W  ...        22.0       1007.7   
1         WNW           44.0        NNW  ...        25.0       1010.6   
2         WSW           46.0          W  ...        30.0       1007.6   
3          NE           24.0         SE  ...        16.0       1017.6   
4           W           41.0        ENE  ...        33.0       1010.8   

   Pressure3pm  Cloud9am  Cloud3pm  Temp9am  Temp3pm  RainToday  RainTomo

# Data Preprocessing

In this step, we need to clean the dataset by dropping redundant columns that do not contribute to the prediction or may cause data leakage. We also need to remove any null values to ensure the model can be trained effectively.

In [ ]:
# Drop redundant columns
df = df.drop(['Date', 'Location', 'RISK_MM'], axis=1)

# Remove null values
df = df.dropna()

print("Cleaned Data Shape:", df.shape)

Cleaned Data Shape: (56420, 21)


# Splitting Data & Building the Model

To build the model, we first isolate the target variable (`RainTomorrow`) from the features. We then perform a 75/25 train-test split to ensure we have a separate set of data to evaluate the model's performance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# Convert categorical variables to dummy/indicator variables
df = pd.get_dummies(df, drop_first=True)

# Isolate target variable
X = df.drop('RainTomorrow_Yes', axis=1)
y = df['RainTomorrow_Yes']

# Train-Test Split (75/25)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize and fit the model
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=5000)

# Evaluation

After training the model, we test its accuracy on the unseen test data to see how well it generalizes to new observations.

In [ ]:
# Make predictions
y_pred = model.predict(X_test)

# Calculate accuracy score
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 85.66%
